In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
df_test = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_test_with_ESM1b_ts.pkl")
)
df_test = df_test.loc[df_test["ESM1b"] != ""]
df_test.reset_index(inplace=True, drop=True)

df_train = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_train_with_ESM1b_ts.pkl")
)
df_train = df_train.loc[df_train["ESM1b"] != ""]
df_train.reset_index(inplace=True, drop=True)

/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)
/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


In [ ]:
p450plant0 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant0.pkl")
)
p450plant1 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant1.pkl")
)
p450plant2 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant2.pkl")
)
p450plant3 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant3.pkl")
)
p450plant4 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant4.pkl")
)

p450plant = pd.concat(
    [p450plant0, p450plant1, p450plant2, p450plant3, p450plant4], ignore_index=True
)

In [4]:
p450plant

,enzyme,substrate,Binding,ECFP,ESM1b
0,CYP716E41,MAS,1,0000000000000000000000000000000001001000000000...,"[-0.1127784475684166, 0.23461320996284485, 0.0..."
1,CYP726A17,CAS,1,0000000000000000000000000100000001001000000000...,"[-0.07275320589542389, 0.12327290326356888, -0..."
2,CYP71D15,LIM,1,0000000000000000000000000000000001000000000000...,"[-0.0860673263669014, 0.16397367417812347, 0.0..."
3,CYP72A66v2,BAM,1,0000000000000000000000000000000101001000000000...,"[-0.0892251580953598, 0.15670408308506012, -0...."
4,CYP76AH15,MOX,1,0000100000000000000000000000000001001000000000...,"[-0.04311390966176987, 0.1504163295030594, -0...."
...,...,...,...,...,...
1223,CYP90G4,CAT,0,0100000010000000000000000000010001001000000000...,"[-0.1249723955988884, 0.2523147463798523, 0.07..."
1224,CYP72A610,NO,0,0000000000000000000000000000000000000000000000...,"[-0.07047156989574432, 0.16309861838817596, 0...."
1225,CYP72A610,FEU,0,0000000000000000000000000100000101000000000000...,"[-0.07047156989574432, 0.16309861838817596, 0...."
1226,CYP79D1,NAR,0,0100000000000000000000000000000000000000000000...,"[-0.01571257971227169, 0.19178493320941925, 0...."


In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    ids = []
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)
        ids.append(ind)
        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y, ids)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [ ]:
train_X, train_y, _ = create_input_and_output_data(df=df_train)
test_X, test_y, _ = create_input_and_output_data(df=df_test)

test_new_X, test_new_y, ids = create_input_and_output_data(df=p450plant)


from sklearn.model_selection import train_test_split

# same split as in Mou et al paper:
X_train_Mou, X_test_Mou, y_train_Mou, y_test_Mou, train_ids, test_ids = (
    train_test_split(test_new_X, test_new_y, ids, test_size=0.20, random_state=42)
)

# data_length = len(test_new_X)
# test_length = int(data_length * 0.2)
# train_length = data_length - test_length

# X_train_Mou = test_new_X[test_length:]
# y_train_Mou = test_new_y[test_length:]

# X_test_Mou = test_new_X[:test_length]
# y_test_Mou = test_new_y[:test_length]

# train_ids = ids[test_length:]
# test_ids = ids[:test_length]


train_X = np.concatenate([train_X, X_train_Mou])
train_y = np.concatenate([train_y, y_train_Mou])

In [7]:
test_ids

[710,
 486,
 244,
 551,
 1162,
 523,
 599,
 447,
 101,
 128,
 1172,
 1093,
 210,
 1049,
 581,
 1018,
 707,
 376,
 1032,
 497,
 1135,
 1003,
 1026,
 377,
 265,
 695,
 451,
 1165,
 49,
 43,
 192,
 352,
 113,
 1099,
 609,
 886,
 1068,
 107,
 390,
 155,
 859,
 1085,
 868,
 458,
 168,
 233,
 902,
 918,
 834,
 908,
 767,
 365,
 306,
 44,
 290,
 506,
 63,
 1214,
 803,
 382,
 1219,
 439,
 1173,
 777,
 355,
 605,
 535,
 678,
 76,
 323,
 1169,
 405,
 78,
 358,
 493,
 420,
 1171,
 243,
 947,
 1177,
 327,
 542,
 70,
 541,
 23,
 1213,
 963,
 58,
 626,
 945,
 485,
 723,
 81,
 371,
 1033,
 174,
 303,
 1124,
 482,
 925,
 438,
 170,
 958,
 667,
 51,
 433,
 1053,
 589,
 1096,
 1180,
 123,
 209,
 1216,
 54,
 808,
 682,
 1084,
 56,
 86,
 585,
 876,
 109,
 549,
 1190,
 836,
 847,
 933,
 422,
 430,
 671,
 367,
 644,
 427,
 428,
 1139,
 423,
 1007,
 911,
 1094,
 999,
 829,
 363,
 294,
 770,
 986,
 701,
 732,
 942,
 1092,
 286,
 398,
 720,
 1195,
 651,
 461,
 490,
 381,
 409,
 1150,
 10,
 147,
 588,
 962,
 95

In [ ]:
param = {
    "learning_rate": 0.31553117247348733,
    "max_delta_step": 1.7726044219753656,
    "max_depth": 10,
    "min_child_weight": 1.3845040588450772,
    "num_rounds": 342.68325188584106,
    "reg_alpha": 0.531395259755843,
    "reg_lambda": 3.744980563764689,
    "weight": 0.26187490421514203,
}

num_round = param["num_rounds"]
# param["tree_method"] = "gpu_hist"
# param["sampling_method"] = "gradient_based"
param["objective"] = "binary:logistic"
param["eval_metric"] = ["error", "logloss"]

weights1 = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in df_train["Binding"]]
)
weights2 = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in y_train_Mou]
)

weights = np.concatenate([weights1, weights2])


del param["num_rounds"]
del param["weight"]

dtrain = xgb.DMatrix(
    np.array(train_X),
    weight=weights,
    label=np.array(train_y),
    feature_names=feature_names,
)
dtest = xgb.DMatrix(
    np.array(test_X), label=np.array(test_y), feature_names=feature_names
)

dtest_new = xgb.DMatrix(
    np.array(X_test_Mou), label=np.array(y_test_Mou), feature_names=feature_names
)

evallist = [(dtest_new, "eval"), (dtrain, "train")]

bst = xgb.train(param, dtrain, int(num_round), evallist, verbose_eval=10)
y_test_pred = np.round(bst.predict(dtest))
acc_test = np.mean(y_test_pred == np.array(test_y))
roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

print("Accuracy on test set: %s, ROC-AUC score for test set: %s" % (acc_test, roc_auc))


y_test_new_pred = np.round(bst.predict(dtest_new))
acc_test_new = np.mean(y_test_new_pred == np.array(y_test_Mou))
try:
    roc_auc_new = roc_auc_score(np.array(y_test_Mou), bst.predict(dtest_new))
    mcc = matthews_corrcoef(np.array(y_test_Mou), y_test_new_pred)
except:
    roc_auc_new = 0
    mcc = 0

print(
    "Accuracy on test set: %s, ROC-AUC score for test set: %s, MCC: %s"
    % (acc_test_new, roc_auc_new, mcc)
)

[0]	eval-error:0.57317	eval-logloss:0.70931	train-error:0.26802	train-logloss:0.62493
[10]	eval-error:0.49594	eval-logloss:0.71875	train-error:0.13072	train-logloss:0.41808
[20]	eval-error:0.44309	eval-logloss:0.69964	train-error:0.08928	train-logloss:0.34519
[30]	eval-error:0.41463	eval-logloss:0.67312	train-error:0.06352	train-logloss:0.29475
[40]	eval-error:0.34959	eval-logloss:0.62668	train-error:0.04812	train-logloss:0.25666
[50]	eval-error:0.35772	eval-logloss:0.61575	train-error:0.03648	train-logloss:0.22421
[60]	eval-error:0.35366	eval-logloss:0.60814	train-error:0.02874	train-logloss:0.19870
[70]	eval-error:0.34146	eval-logloss:0.59448	train-error:0.02279	train-logloss:0.17590
[80]	eval-error:0.32927	eval-logloss:0.58033	train-error:0.01858	train-logloss:0.15818
[90]	eval-error:0.30894	eval-logloss:0.56608	train-error:0.01587	train-logloss:0.14385
[100]	eval-error:0.30081	eval-logloss:0.56504	train-error:0.01337	train-logloss:0.12973
[110]	eval-error:0.30081	eval-logloss:0.561

In [ ]:
index_of_ones = np.where(np.array(y_test_Mou) == 1)[0]
values_of_ones = bst.predict(dtest_new)[index_of_ones]
acc_1 = np.mean(np.round(values_of_ones) == 1)

index_of_zeros = np.where(np.array(y_test_Mou) == 0)[0]
values_of_zeros = bst.predict(dtest_new)[index_of_zeros]
acc_0 = np.mean(np.round(values_of_zeros) == 0)

print("Accuracy on 1 set: %s, Accuracy on 0 set: %s" % (acc_1, acc_0))
print(len(train_y))
print(len(y_test_Mou))

Accuracy on 1 set: 0.5925925925925926, Accuracy on 0 set: 0.793939393939394
56959
246


In [ ]:
pickle.dump(bst, open(join(our_data + "p450normalmodel.dat"), "wb"))

In [11]:
p450plant

,enzyme,substrate,Binding,ECFP,ESM1b
0,CYP716E41,MAS,1,0000000000000000000000000000000001001000000000...,"[-0.1127784475684166, 0.23461320996284485, 0.0..."
1,CYP726A17,CAS,1,0000000000000000000000000100000001001000000000...,"[-0.07275320589542389, 0.12327290326356888, -0..."
2,CYP71D15,LIM,1,0000000000000000000000000000000001000000000000...,"[-0.0860673263669014, 0.16397367417812347, 0.0..."
3,CYP72A66v2,BAM,1,0000000000000000000000000000000101001000000000...,"[-0.0892251580953598, 0.15670408308506012, -0...."
4,CYP76AH15,MOX,1,0000100000000000000000000000000001001000000000...,"[-0.04311390966176987, 0.1504163295030594, -0...."
...,...,...,...,...,...
1223,CYP90G4,CAT,0,0100000010000000000000000000010001001000000000...,"[-0.1249723955988884, 0.2523147463798523, 0.07..."
1224,CYP72A610,NO,0,0000000000000000000000000000000000000000000000...,"[-0.07047156989574432, 0.16309861838817596, 0...."
1225,CYP72A610,FEU,0,0000000000000000000000000100000101000000000000...,"[-0.07047156989574432, 0.16309861838817596, 0...."
1226,CYP79D1,NAR,0,0100000000000000000000000000000000000000000000...,"[-0.01571257971227169, 0.19178493320941925, 0...."


In [12]:
xtracted_rows = p450plant.loc[test_ids]

In [ ]:
xtracted_rows_group = xtracted_rows.groupby("enzyme", as_index=False).count()

In [ ]:
xtracted_rows_group.to_csv("bin_test920.csv")

In [ ]:
xtracted_rows[xtracted_rows["enzyme"] == "CYP79F2"]

,enzyme,substrate,Binding,ECFP,ESM1b
1093,CYP79F2,FOR,0,0000000000000000000000000000000001000000000000...,"[-0.01744970679283142, 0.20131497085094452, 0...."
1092,CYP79F2,ABA,0,0000100000000000000000000000000001001000000000...,"[-0.01744970679283142, 0.20131497085094452, 0...."
996,CYP79F2,Hexahomomethionine,1,0100000000010000000000000000000001000000000000...,"[-0.01744970679283142, 0.20131497085094452, 0...."
